# ReLLa-lite QLoRA continuation: +10k train examples from 20k to 30k, no evaluation

This notebook continues training the already continued ReLLa-lite adapter.

Goal:

```text
ReLLa-lite adapter exposed to 20k examples
+ 10k new disjoint train examples
= ReLLa-lite adapter exposed to 30k examples in total
```

Validation/test evaluation is handled after saving the new adapter.

## Input artifacts

### 1. Retrieved instruction dataset

Expected files:

```text
train.jsonl
val.jsonl
test.jsonl
```

The dataset corresponds to `instruction_temporal_retrieved_topk`. Only `train.jsonl` is required for this training run.

### 2. Existing ReLLa-lite 20k adapter

Expected adapter directory:

```text
adapter_qwen_rella_lite_continued_20k/
  adapter_config.json
  adapter_model.safetensors
  tokenizer.json
  tokenizer_config.json
  README.md
```

A zipped adapter archive can also be used:

```text
adapter_qwen_rella_lite_continued_20k.zip
```


In [ ]:
!pip install -q -U "transformers>=4.45.0" accelerate peft bitsandbytes scikit-learn pandas pyarrow safetensors

In [ ]:
import os, json, glob, math, random, shutil, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, Trainer, TrainingArguments
from peft import PeftModel, prepare_model_for_kbit_training

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("device count:", torch.cuda.device_count())

## 1. Configuration

Previous ReLLa-lite exposure:

```text
first chunk:  5k Yes + 5k No
second chunk: 5k Yes + 5k No
total before this notebook: 20k examples
```

This notebook adds:

```text
third chunk: 5k Yes + 5k No
```

Final exposure estimate:

```text
30k examples = 15k Yes + 15k No
```


In [ ]:
MODEL_NAME_FALLBACK = "Qwen/Qwen3-4B-Instruct-2507"
MAX_SEQ_LENGTH = 1024

# This reconstructs the previous two chunks and selects a new disjoint third chunk.
FIRST_PER_CLASS = 5000
SECOND_PER_CLASS = 5000
NEW_PER_CLASS = 5000

NUM_EPOCHS = 1
LR = 1e-4
GRAD_ACCUM = 8
BATCH_SIZE = 1
SAVE_STEPS = 500

RUN_NAME = f"qwen_rella_lite_continue_new{NEW_PER_CLASS*2}_from20k_no_eval"
OUT_DIR = Path("/kaggle/working") / RUN_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("RUN_NAME:", RUN_NAME)
print("OUT_DIR:", OUT_DIR)

## 2. Locate train.jsonl and the 20k adapter

In [ ]:
def find_file(filename, prefer_keywords=None, avoid_keywords=None):
    hits = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    hits += glob.glob(f"/kaggle/working/unzipped_inputs/**/{filename}", recursive=True)
    if not hits:
        raise FileNotFoundError(f"Required file was not found: {filename}")

    prefer_keywords = [x.lower() for x in (prefer_keywords or [])]
    avoid_keywords = [x.lower() for x in (avoid_keywords or [])]

    def score(path):
        p = path.lower()
        prefer_penalty = sum(kw not in p for kw in prefer_keywords)
        avoid_penalty = sum(kw in p for kw in avoid_keywords)
        return (prefer_penalty, avoid_penalty, len(path))

    return Path(sorted(hits, key=score)[0])

def unzip_relevant_inputs():
    dst_root = Path("/kaggle/working/unzipped_inputs")
    dst_root.mkdir(parents=True, exist_ok=True)

    zip_hits = glob.glob("/kaggle/input/**/*.zip", recursive=True)
    for z in zip_hits:
        name = os.path.basename(z).lower()
        if any(k in name for k in ["rella", "adapter", "continued", "20k"]):
            dst = dst_root / Path(z).stem
            dst.mkdir(parents=True, exist_ok=True)
            try:
                with zipfile.ZipFile(z, "r") as f:
                    f.extractall(dst)
                print("Unzipped:", z, "->", dst)
            except Exception as e:
                print("Could not unzip:", z, e)

unzip_relevant_inputs()

train_path = find_file("train.jsonl", prefer_keywords=["retrieved", "topk", "instruction", "rella"])

adapter_config_path = find_file(
    "adapter_config.json",
    prefer_keywords=["rella", "continued", "20k", "adapter"],
    avoid_keywords=["train10000", "10k", "checkpoint3000", "checkpoint2500", "ordinary"]
)
adapter_dir = adapter_config_path.parent

print("train_path:", train_path)
print("adapter_dir:", adapter_dir)
print("adapter files:", [p.name for p in adapter_dir.iterdir() if p.is_file()][:30])

if "20" not in str(adapter_dir).lower() and "continued" not in str(adapter_dir).lower():
    print("WARNING: adapter_dir does not clearly look like the continued 20k adapter. Manual path check is required.")

In [ ]:
with open(adapter_config_path, "r", encoding="utf-8") as f:
    adapter_cfg = json.load(f)

MODEL_NAME = adapter_cfg.get("base_model_name_or_path") or MODEL_NAME_FALLBACK
if MODEL_NAME.startswith("/") or MODEL_NAME.startswith("./"):
    MODEL_NAME = MODEL_NAME_FALLBACK

print("MODEL_NAME:", MODEL_NAME)
print("adapter config subset:")
print(json.dumps({
    "base_model_name_or_path": adapter_cfg.get("base_model_name_or_path"),
    "peft_type": adapter_cfg.get("peft_type"),
    "task_type": adapter_cfg.get("task_type"),
    "r": adapter_cfg.get("r"),
    "lora_alpha": adapter_cfg.get("lora_alpha"),
    "target_modules": adapter_cfg.get("target_modules"),
}, ensure_ascii=False, indent=2))

## 3. Load data and select the third disjoint +10k chunk

This cell reconstructs the previous two chunks:

```text
old first 10k = random_state 42
old second 10k = random_state 142 from the remaining data
new third 10k = random_state 242 from the remaining data
```

This matches the previous continuation notebook logic and keeps the new chunk disjoint from the old 20k examples.


In [ ]:
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return pd.DataFrame(rows)

train_df = load_jsonl(train_path)
print("train shape:", train_df.shape)
print("output distribution:", train_df["output"].value_counts(dropna=False).to_dict())

def class_mask(df, cls):
    return df["output"].astype(str).str.strip().str.lower() == cls.lower()

def make_third_continuation_chunk(df, first_per_class, second_per_class, new_per_class, seed=42):
    yes = df[class_mask(df, "yes")]
    no = df[class_mask(df, "no")]

    required = first_per_class + second_per_class + new_per_class
    if len(yes) < required:
        raise ValueError(f"Not enough Yes examples: have {len(yes)}, need {required}.")
    if len(no) < required:
        raise ValueError(f"Not enough No examples: have {len(no)}, need {required}.")

    first_yes_idx = yes.sample(n=first_per_class, random_state=seed).index
    first_no_idx = no.sample(n=first_per_class, random_state=seed).index

    yes_after_first = yes.drop(index=first_yes_idx)
    no_after_first = no.drop(index=first_no_idx)

    second_yes_idx = yes_after_first.sample(n=second_per_class, random_state=seed + 100).index
    second_no_idx = no_after_first.sample(n=second_per_class, random_state=seed + 100).index

    yes_after_second = yes_after_first.drop(index=second_yes_idx)
    no_after_second = no_after_first.drop(index=second_no_idx)

    third_yes = yes_after_second.sample(n=new_per_class, random_state=seed + 200)
    third_no = no_after_second.sample(n=new_per_class, random_state=seed + 200)

    chunk = pd.concat([third_yes, third_no], axis=0)
    chunk = chunk.sample(frac=1.0, random_state=seed + 201).reset_index(drop=True)

    diagnostics = {
        "first_yes": int(len(first_yes_idx)),
        "first_no": int(len(first_no_idx)),
        "second_yes": int(len(second_yes_idx)),
        "second_no": int(len(second_no_idx)),
        "new_third_yes": int(len(third_yes)),
        "new_third_no": int(len(third_no)),
        "remaining_yes_after_previous_20k": int(len(yes_after_second)),
        "remaining_no_after_previous_20k": int(len(no_after_second)),
    }

    if "sample_id" in df.columns:
        previous_ids = set(df.loc[
            list(first_yes_idx) + list(first_no_idx) + list(second_yes_idx) + list(second_no_idx),
            "sample_id"
        ].astype(str).tolist())
        new_ids = set(chunk["sample_id"].astype(str).tolist())
        diagnostics["sample_id_overlap_with_previous_20k"] = int(len(previous_ids.intersection(new_ids)))
        diagnostics["new_unique_sample_ids"] = int(len(new_ids))

    return chunk, diagnostics

train_chunk_df, chunk_diagnostics = make_third_continuation_chunk(
    train_df,
    FIRST_PER_CLASS,
    SECOND_PER_CLASS,
    NEW_PER_CLASS,
    SEED
)

print("train chunk:", train_chunk_df.shape)
print("chunk output distribution:", train_chunk_df["output"].value_counts().to_dict())
print("chunk diagnostics:")
print(json.dumps(chunk_diagnostics, ensure_ascii=False, indent=2))

assert chunk_diagnostics.get("sample_id_overlap_with_previous_20k", 0) == 0, "New chunk overlaps with previous 20k examples."
assert len(train_chunk_df) == 2 * NEW_PER_CLASS, "Unexpected train chunk size."


## 4. Tokenizer and dataset

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

SYSTEM_PROMPT = "You are a recommendation model. Answer only Yes or No."

def normalize_answer(x):
    x = str(x).strip()
    if x.lower().startswith("yes"):
        return "Yes"
    if x.lower().startswith("no"):
        return "No"
    raise ValueError(f"Unexpected answer value: {x}")

def build_user_text(row):
    instruction = str(row.get("instruction", "")).strip()
    inp = str(row.get("input", "")).strip()
    if instruction and inp:
        return instruction + "\n\n" + inp
    if instruction:
        return instruction
    if "prompt_text" in row and pd.notna(row["prompt_text"]):
        return str(row["prompt_text"]).strip()
    raise ValueError("The row does not contain instruction/input fields required for prompt construction.")

def make_prompt_text(row):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_text(row)}
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def make_answer_text(row):
    return normalize_answer(row["output"]) + tokenizer.eos_token

print(make_prompt_text(train_chunk_df.iloc[0])[:2500])
print("answer:", make_answer_text(train_chunk_df.iloc[0]))

In [ ]:
class RecSFTDataset(torch.utils.data.Dataset):
    def __init__(self, df, tokenizer, max_length=1024):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt_ids = self.tokenizer(make_prompt_text(row), add_special_tokens=False).input_ids
        answer_ids = self.tokenizer(make_answer_text(row), add_special_tokens=False).input_ids

        max_prompt_len = self.max_length - len(answer_ids)
        if len(prompt_ids) > max_prompt_len:
            prompt_ids = prompt_ids[-max_prompt_len:]

        input_ids = prompt_ids + answer_ids
        labels = [-100] * len(prompt_ids) + answer_ids
        attention_mask = [1] * len(input_ids)

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }

def collate_fn(batch):
    max_len = max(len(x["input_ids"]) for x in batch)
    input_ids, attention_mask, labels = [], [], []

    for x in batch:
        pad_len = max_len - len(x["input_ids"])
        input_ids.append(torch.cat([x["input_ids"], torch.full((pad_len,), tokenizer.pad_token_id, dtype=torch.long)]))
        attention_mask.append(torch.cat([x["attention_mask"], torch.zeros(pad_len, dtype=torch.long)]))
        labels.append(torch.cat([x["labels"], torch.full((pad_len,), -100, dtype=torch.long)]))

    return {
        "input_ids": torch.stack(input_ids),
        "attention_mask": torch.stack(attention_mask),
        "labels": torch.stack(labels),
    }

train_ds = RecSFTDataset(train_chunk_df, tokenizer, MAX_SEQ_LENGTH)
lens = [len(train_ds[i]["input_ids"]) for i in range(min(1000, len(train_ds)))]
print("length min/mean/max:", min(lens), float(np.mean(lens)), max(lens))
print("train examples:", len(train_ds))

## 5. Load base model and trainable existing adapter

In [ ]:
compute_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

base_model.config.use_cache = False
base_model = prepare_model_for_kbit_training(base_model)

model = PeftModel.from_pretrained(base_model, str(adapter_dir), is_trainable=True)
model.print_trainable_parameters()
print("compute_dtype:", compute_dtype)

## 6. Continue training and save adapter immediately

In [ ]:
training_args = TrainingArguments(
    output_dir=str(OUT_DIR / "trainer"),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    fp16=(compute_dtype == torch.float16),
    bf16=(compute_dtype == torch.bfloat16),
    logging_steps=20,
    save_steps=SAVE_STEPS,
    save_strategy="steps",
    save_total_limit=3,
    report_to="none",
    optim="paged_adamw_8bit",
    seed=SEED,
    remove_unused_columns=False,
    gradient_checkpointing=True,
    warmup_ratio=0.03,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    data_collator=collate_fn,
)

approx_steps = math.ceil(len(train_ds) / (BATCH_SIZE * GRAD_ACCUM)) * NUM_EPOCHS
print("approx optimization steps:", approx_steps)

trainer.train()

adapter_out_dir = OUT_DIR / "adapter_qwen_rella_lite_continued_30k"
model.save_pretrained(adapter_out_dir)
tokenizer.save_pretrained(adapter_out_dir)
archive_path = shutil.make_archive(str(OUT_DIR / "adapter_qwen_rella_lite_continued_30k"), "zip", adapter_out_dir)

run_summary = {
    "model_name": MODEL_NAME,
    "source_adapter_dir": str(adapter_dir),
    "source_adapter_stage": "rella_lite_continued_20k",
    "source_dataset": str(train_path),
    "dataset": "instruction_temporal_retrieved_topk",
    "first_per_class": int(FIRST_PER_CLASS),
    "second_per_class": int(SECOND_PER_CLASS),
    "new_per_class": int(NEW_PER_CLASS),
    "new_train_examples": int(len(train_chunk_df)),
    "total_exposed_examples_estimate": int(2 * FIRST_PER_CLASS + 2 * SECOND_PER_CLASS + 2 * NEW_PER_CLASS),
    "num_epochs_on_new_chunk": int(NUM_EPOCHS),
    "learning_rate": float(LR),
    "max_seq_length": int(MAX_SEQ_LENGTH),
    "batch_size": int(BATCH_SIZE),
    "gradient_accumulation_steps": int(GRAD_ACCUM),
    "chunk_diagnostics": chunk_diagnostics,
    "output_adapter_dir": str(adapter_out_dir),
    "adapter_zip": str(archive_path),
}

summary_path = OUT_DIR / "qwen_rella_lite_continued_30k_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(run_summary, f, ensure_ascii=False, indent=2)

print("saved:", OUT_DIR)
print("adapter folder:", adapter_out_dir)
print("adapter zip:", archive_path)
print("summary:", summary_path)

## Output artifacts

Main artifacts:

```text
adapter_qwen_rella_lite_continued_30k.zip
qwen_rella_lite_continued_30k_summary.json
```

Evaluation input options:

```text
adapter_qwen_rella_lite_continued_30k.zip
```

or an adapter directory:

```text
adapter_qwen_rella_lite_continued_30k/
  adapter_config.json
  adapter_model.safetensors
  tokenizer.json
  tokenizer_config.json
  README.md
```
